### Identify the intersection between fetal-enriched and disease-enriched TFs

In [6]:
library(tidyverse)

In [11]:
fetal_res <- read.csv("04B_fetal_sig_regulon_df.csv")
fetal_res <- fetal_res %>%
  mutate(regulon = str_extract(regulon, "^[^_]+"))
fetal_res <- fetal_res %>% dplyr::filter(padj < 0.05)

disease_res <- read.csv("04B_disease_sig_regulon_df.csv")
disease_res <- disease_res %>%
  mutate(regulon = str_extract(regulon, "^[^_]+"))
disease_res <- disease_res %>% dplyr::filter(padj < 0.05)

In [12]:
fetal_df <- fetal_res %>%
  select(regulon, cell_type, log2FC_fetal = log2FC)

disease_df <- disease_res %>%
  select(regulon, cell_type, log2FC_disease = log2FC)

In [13]:
shared_regulons <- fetal_df %>%
  inner_join(disease_df, by = c("regulon", "cell_type"))

shared_regulons_dir <- shared_regulons %>%
  filter(sign(log2FC_fetal) == sign(log2FC_disease))

In [14]:
shared_regulons_dir

regulon,cell_type,log2FC_fetal,log2FC_disease
<chr>,<chr>,<dbl>,<dbl>
AR,Cardiomyocyte,-1.8722039,-0.06106599
EBF3,Cardiomyocyte,0.7311192,0.09819716
TCF4,Cardiomyocyte,0.6855273,0.10440702
KLF12,Cardiomyocyte,0.3444485,0.06749628
ZNF704,Cardiomyocyte,0.4387626,0.03795624
ZNF90,Cardiomyocyte,2.1582879,0.41869614
RUNX1,Cardiomyocyte,0.3637942,0.02570023
ZNF846,Cardiomyocyte,0.7189348,0.17497937
BNC2,Cardiomyocyte,0.3827858,0.10150904


In [15]:
sig_shared_regulons_dir <- shared_regulons_dir %>%
  filter(abs(log2FC_fetal) > 0.5, abs(log2FC_disease) > 0.1)

In [23]:
df_long <- sig_shared_regulons_dir %>%
  pivot_longer(
    cols = starts_with("log2FC_"),
    names_to = "comparison",
    names_prefix = "log2FC_",
    values_to = "log2FC"
  )

df_long %>% head()

regulon,cell_type,comparison,log2FC
<chr>,<chr>,<chr>,<dbl>
TCF4,Cardiomyocyte,fetal,0.6855273
TCF4,Cardiomyocyte,disease,0.1044070
ZNF90,Cardiomyocyte,fetal,2.1582879
ZNF90,Cardiomyocyte,disease,0.4186961
ZNF846,Cardiomyocyte,fetal,0.7189348
ZNF846,Cardiomyocyte,disease,0.1749794


In [42]:
unique(df_long$regulon)

[1] "TCF4"   "ZNF90"  "ZNF846" "PLAGL1" "NFIX"   "ZNF423" "JUNB"   "FOSB"  
 [9] "PGR"    "AR"     "RUNX2"  "ZNF680" "MTA3"   "CEBPD"  "CREB5"  "NFE2L3"
[17] "SOX5"   "PDLIM5" "BACH2"  "KLF3"

In [41]:
options(repr.plot.width = 15, repr.plot.height = 6)

p1 <- ggplot(data = df_long, aes(x = comparison, y = regulon, size = log2FC, color = log2FC)) + 
    geom_point() +
    facet_wrap(~ cell_type, ncol = 5) +  
    scale_color_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) + 
    theme_bw() +
    theme(
        strip.background = element_blank(),  
        strip.text = element_text(size = 14, color = "black"),  # clean facet labels
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14),
        axis.text.x = element_text(size = 12, angle = 45, hjust = 1),
        axis.text.y = element_text(size = 12),
        axis.title.x = element_blank(),
        axis.title.y = element_blank(),
        plot.title = element_text(size = 18, hjust = 0.5)
)

ggsave(p1, filename = "05_fetal_reactivation_TFs.pdf", width = 15, height = 6)